In [1]:
import numpy as np
from sklearn.datasets import load_diabetes

import warnings
warnings.filterwarnings('ignore')

In [2]:
X,y = load_diabetes(return_X_y = True)
X.shape

(442, 10)

In [3]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state = 42)
X_train.shape, y_train.shape

((353, 10), (353,))

In [4]:
X_test.shape, y_test.shape

((89, 10), (89,))

In [6]:
from sklearn.linear_model import Ridge

ridge_reg = Ridge(alpha = 0.1, solver = 'cholesky')
ridge_reg.fit(X_train, y_train)

ridge_reg.intercept_, ridge_reg.coef_

(np.float64(151.45857456679613),
 array([  42.85566976, -205.49431899,  505.08903304,  317.0932049 ,
        -108.50026183,  -86.23673333, -190.36318008,  151.70708637,
         392.28931896,   79.9081772 ]))

In [7]:
y_prediction = ridge_reg.predict(X_test)

from sklearn.metrics import r2_score
r2_score(y_test, y_prediction)

0.46085219464119265

### Ridge Regression Using Our Own Class

In [17]:
class RidgeRegression:
    def __init__(self, alpha = 0.1):
        self.alpha = alpha
        self.intercept_ = None
        self.coef_ = None

    def fit(self, X_train, y_train):
        X_train = np.insert(X_train, 0, 1, axis = 1)
        A = np.eye(X_train.shape[1])
        A[0][0] = 0

        weights = np.linalg.inv(np.dot(X_train.T, X_train) + self.alpha*A).dot(np.dot(X_train.T,y_train))
        self.intercept_ = weights[0]
        self.coef_ = weights[1:]

    def predict(self, X_test):
        return np.dot(X_test, self.coef_) + self.intercept_

In [20]:
ridge_reg = RidgeRegression(alpha = 0.1)
ridge_reg.fit(X_train, y_train)

ridge_reg.intercept_, ridge_reg.coef_

(np.float64(151.45857456679616),
 array([  42.85566976, -205.49431899,  505.08903304,  317.0932049 ,
        -108.50026183,  -86.23673333, -190.36318008,  151.70708637,
         392.28931896,   79.9081772 ]))

In [21]:
y_prediction = ridge_reg.predict(X_test)
r2_score(y_test, y_prediction)

0.46085219464119254

### Ridge Regression Using Gradient Descent

In [25]:
from sklearn.linear_model import SGDRegressor
ridge_reg_sgd = SGDRegressor(penalty = 'l2', max_iter = 500, eta0 = 0.1, learning_rate = 'constant', alpha = 0.001)
ridge_reg_sgd.fit(X_train, y_train)

ridge_reg_sgd.intercept_, ridge_reg_sgd.coef_

(array([149.28553275]),
 array([  48.70376739, -134.65897274,  395.74478401,  265.22730203,
         -25.42256233,  -65.88095614, -181.44200269,  137.61673732,
         300.36131335,  116.57236486]))

In [26]:
# Using Ridge class
ridge = Ridge(alpha = 0.001, max_iter = 500, solver = 'sparse_cg')
ridge.fit(X_train, y_train)
ridge.intercept_, ridge.coef_

(np.float64(151.3419806004085),
 array([  38.48350222, -241.35179746,  543.83517435,  346.78282693,
        -827.70199985,  437.17374349,  116.9497567 ,  260.75925802,
         696.12921785,   49.74344197]))

In [28]:
y_hat = ridge.predict(X_test)
r2_score(y_test, y_hat)

0.4534280305933893

### Using Manual Class

In [37]:
class RdigeSGD:
    def __init__(self, alpha, learning_rate, epoch):
        self.alpha = alpha
        self.learning_rate = learning_rate
        self.epoch = epoch
        self.intercept_ =  None
        self.coef_ = None

    def fit(self, X_train, y_train):
        self.coef_ = np.ones(X_train.shape[1])
        self.intercept_ = 0
        X_train = np.insert(X_train, 0, 1, axis = 1)
        weights = np.insert(self.coef_, 0, self.intercept_)

        for i in range(self.epoch):
            gradient = np.dot(np.dot(X_train.T, X_train), weights) - np.dot(X_train.T, y_train) + self.alpha*weights
            weights = weights - self.learning_rate*gradient

        self.coef_ = weights[1:]
        self.intercept_ = weights[0]

    def predict(self, X_test):
        return np.dot(X_test, self.coef_) + self.intercept_
        

In [44]:
rsgd = RdigeSGD(alpha = 0.001, learning_rate = 0.005, epoch = 500)
rsgd.fit(X_train, y_train)
rsgd.intercept_, rsgd.coef_

(np.float64(151.40454755919714),
 array([  43.37183861, -192.03766574,  496.43542567,  319.37407   ,
         -64.42788084, -113.194338  , -213.9073644 ,  144.86136322,
         367.67948022,  119.56857869]))

In [45]:
y_pred = rsgd.predict(X_test)
r2_score(y_test,y_pred)

0.4592249376936406